In [2]:
#@title student data
from IPython.display import Javascript
colab_base = "https://colab.research.google.com/drive/1BVBGdNkEuqIgFp6XxmaV9NJy5c2sX3ln?usp=sharing"
Grupa = "niedotyczy" # @param ["niedotyczy", "13_45","15_30","17_15"]
Student_ID = "473616" # @param {"type":"string"}
Link_to_this_colab = "https://github.com/pawelFelcyn/nlp" # @param {"type":"string"}
Mail = "" # @param {"type":"string","placeholder":"Optional"}

if Link_to_this_colab == colab_base:
    print("Podaj link do swojego Colab")
    display(Javascript('alert("Podaj link do swojego Colab");'))

#Task C01

## 🧩 Skalarowy produkt uwagi (scaled dot-product attention) — pojedynczy token

W tym zadaniu zaimplementujesz **podstawowy blok attention** dla pojedynczego tokenu:
policzysz wektory uwagi na podstawie zapytań (`Q`), kluczy (`K`) i wartości (`V`).

---

## 🎯 Cel

- przećwiczyć formułę *scaled dot-product attention* na małym przykładzie,
- zrozumieć, jak z `Q`, `K`, `V` powstają wektory wyjściowe.

---

## 📘 Kontekst

Dla jednego headu attention (bez podziału na heady) formuła jest:

$$
\text{Attention}(q, K, V) = \text{softmax}\left( \frac{q K^T}{\sqrt{d_k}} \right) V
$$

gdzie:

- `q` ma wymiar \(d_k\),
- `K` ma wymiar `(seq_len, d_k)`,
- `V` ma wymiar `(seq_len, d_v)`.

W tym zadaniu pracujemy na **jednym zapytaniu** `q` i całym `K`, `V`.

---

## ✅ Twoje zadanie

1. W komórce `dane` masz zdefiniowane:
   - `q` — wektor zapytania (kształt `(4,)`),
   - `K` — macierz kluczy (kształt `(3, 4)`),
   - `V` — macierz wartości (kształt `(3, 2)`).
2. Zaimplementuj funkcję:

   ```python
   def scaled_dot_product_attention_single(q, K, V):
       ...
   ```

   która:
   - oblicza wektor logitów: `scores = q @ K.T`,
   - skaluje go przez `np.sqrt(d_k)`,
   - stosuje `softmax` po osi 0,
   - oblicza wynik: ważoną sumę wierszy `V` z wagami `softmax(scores)`.
3. W komórce `Answer`:
   - wywołaj `scaled_dot_product_attention_single(q, K, V)`,
   - wynikowy wektor zapisz jako string w `final_answer`:

   - zaokrąglij wartości do **3 miejsc po przecinku**,
   - liczby oddziel przecinkami, **bez spacji**, np.:

   ```text
   0.123,-0.456,1.000
   ```

`final_answer` **musi być dokładnie takim stringiem**, bo serwer porówna go z kluczem.


### 🔗 Przydatne funkcje NumPy:
- `np.exp` — https://numpy.org/doc/stable/reference/generated/numpy.exp.html
- `np.sqrt` — https://numpy.org/doc/stable/reference/generated/numpy.sqrt.html
- `np.sum` — https://numpy.org/doc/stable/reference/generated/numpy.sum.html
- Operacje macierzowe — https://numpy.org/doc/stable/user/basics.broadcasting.html

### 📝 O softmax

Softmax zamienia wektor liczb na rozkład prawdopodobieństwa — wszystkie wartości są dodatnie i sumują się do 1.  
W praktyce stosujemy **wersję numerycznie stabilną**, odejmując `max` od wektora przed exponentem.

Nie ma gotowej funkcji `np.softmax` — dlatego trzeba zaimplementować ją ręcznie, używając:

- `np.exp`: https://numpy.org/doc/stable/reference/generated/numpy.exp.html  
- `np.sum`: https://numpy.org/doc/stable/reference/generated/numpy.sum.html  
- opis stabilnego softmax: https://cs231n.github.io/linear-classify/#softmax  


In [3]:
#@title dane

import numpy as np

# wymiar d_k = 4, d_v = 2
q = np.array([0.5, 1.0, -0.5, 0.0], dtype=float)

K = np.array([
    [1.0, 0.0, 0.0, 0.0],
    [0.0, 1.0, 0.5, 0.0],
    [0.5, -0.5, 0.0, 1.0],
], dtype=float)

V = np.array([
    [1.0, 2.0],
    [0.0, 1.0],
    [-1.0, 0.5],
], dtype=float)


In [6]:
#@title code

import numpy as np

def softmax(x):
    """Oblicza softmax po wektorze 1D w sposób numerycznie stabilny."""
    x = np.asarray(x, dtype=float)
    x_shifted = x - np.max(x)
    exps = np.exp(x_shifted)
    return exps / np.sum(exps)

def scaled_dot_product_attention_single(q, K, V):
    """Zwraca wektor Attention(q, K, V) dla pojedynczego zapytania q.

    q: wektor (d_k,)
    K: macierz (seq_len, d_k)
    V: macierz (seq_len, d_v)
    """
    scores = np.dot(K, q)
    d = K.shape[1]
    scores /= np.sqrt(d)
    weights = softmax(scores)
    attention = np.dot(weights, V)
    return attention


In [8]:
# @title Answer

att = scaled_dot_product_attention_single(q, K, V)
rounded = [f"{x:.3f}" for x in att]
final_answer = ",".join(rounded)
print(final_answer)


0.111,1.233


In [9]:
#@title Submit answer (final_answer)
import requests
taskID = "TaskC01"
url = 'https://www.duszekjk.com/ugpt/api/submit_answer/'
final_answer_size = len(final_answer)

final_answer_send = final_answer[:300] + "\n" + str(final_answer_size) + " znaków"
print(final_answer_send)
data = {
    'student_id': Student_ID,
    'student_mail': Mail,
    'task': taskID,
    'grupa': Grupa,
    'answer': final_answer_send,
    'share_link': Link_to_this_colab
}

if Link_to_this_colab == colab_base or Link_to_this_colab == "":
    print("Podaj link do swojego Colab")
    display(Javascript('alert("Podaj link do swojego Colab");'))
else:
    if final_answer == "":
        print("Podaj rozwiązanie")
        display(Javascript('alert("Podaj rozwiązanie");'))
    else:
        response = requests.post(url, json=data)
        if response.status_code == 200:
            response_data = response.json()
            print("Wysłano!")

            points = response_data.get('points', 0)
            if points > 0:
                print(f"Punkty za zadanie: {points}")
            else:
                print("Zadanie wymaga ręcznego zatwierdzenia lub wynik jest niepoprawny")
        else:
            print("błąd wysyłki:", response.status_code)
            print("Response Text:", response.text)


0.111,1.233
11 znaków
Wysłano!
Punkty za zadanie: 1


#Task C02

## 🧩 Pełna macierz self-attention dla krótkiej sekwencji

W tym zadaniu policzysz **pełną macierz wag attention** dla trzech tokenów w jednym headzie.

---

## 🎯 Cel

- przećwiczyć obliczanie macierzy `Q K^T`,
- zastosować skalowanie \(1/\sqrt{d_k}\) i softmax po wierszach,
- zrozumieć, jak wygląda macierz wag attention.

---

## 📘 Kontekst

Dla self-attention w jednym headzie:

$$
A = \text{softmax}\left( \frac{Q K^T}{\sqrt{d_k}} \right)
$$

gdzie:

- `Q` ma kształt `(seq_len, d_k)`,
- `K` ma kształt `(seq_len, d_k)`,
- `A` ma kształt `(seq_len, seq_len)`.

Softmax jest liczony **osobno dla każdego wiersza** (po kluczach).

---

## ✅ Twoje zadanie

1. W komórce `dane` masz zdefiniowane:
   - `Q` i `K` o kształcie `(3, 4)`.
2. Zaimplementuj funkcję:

   ```python
   def attention_weights(Q, K):
       ...
   ```

   która:
   - liczy macierz `scores = Q @ K.T`,
   - skaluje ją przez `np.sqrt(d_k)`,
   - stosuje softmax po osi 1 (dla każdego wiersza).
3. W komórce `Answer`:

   - policz `A = attention_weights(Q, K)`,
   - spłaszcz macierz do wektora (row-major, najpierw cały wiersz 0, potem 1, potem 2),
   - zaokrąglij do 3 miejsc po przecinku,
   - zapisz jako string w `final_answer`, np.:

   ```text
   0.111,0.222,0.667,0.100,0.200,0.700,0.300,0.300,0.400
   ```

Czyli **9 liczb** oddzielonych przecinkami, bez spacji.


### 🔗 Przydatne funkcje NumPy:
- `np.matmul` / operator `@` — https://numpy.org/doc/stable/reference/generated/numpy.matmul.html
- `np.exp` — https://numpy.org/doc/stable/reference/generated/numpy.exp.html
- Softmax (implementacja własna) — przykład: https://numpy.org/doc/stable/

In [10]:
#@title dane

import numpy as np

Q = np.array([
    [0.5,  1.0,  0.0, -0.5],
    [1.0,  0.0,  0.5,  0.0],
    [0.0, -0.5,  1.0,  0.5],
], dtype=float)

K = np.array([
    [0.5,  0.5,  0.0,  0.0],
    [1.0,  0.0, -0.5,  0.5],
    [0.0, -0.5,  1.0,  0.0],
], dtype=float)


In [11]:
#@title code

import numpy as np

def softmax_rows(x):
    """Softmax po wierszach macierzy 2D."""
    x = np.asarray(x, dtype=float)
    x_shifted = x - np.max(x, axis=1, keepdims=True)
    exps = np.exp(x_shifted)
    return exps / np.sum(exps, axis=1, keepdims=True)

def attention_weights(Q, K):
    """Zwraca macierz wag attention A o kształcie (seq_len, seq_len)."""
    scores = np.matmul(Q, K.T)
    d = K.shape[1]
    scores /= np.sqrt(d)
    weights = softmax_rows(scores)
    return weights


In [12]:
# @title Answer

A = attention_weights(Q, K)
flat = A.reshape(-1)
rounded = [f"{x:.3f}" for x in flat]
final_answer = ",".join(rounded)
print(final_answer)


0.432,0.337,0.231,0.319,0.362,0.319,0.243,0.243,0.514


In [13]:
#@title Submit answer (final_answer)
import requests
taskID = "TaskC02"
url = 'https://www.duszekjk.com/ugpt/api/submit_answer/'
final_answer_size = len(final_answer)

final_answer_send = final_answer[:300] + "\n" + str(final_answer_size) + " znaków"
print(final_answer_send)
data = {
    'student_id': Student_ID,
    'student_mail': Mail,
    'task': taskID,
    'grupa': Grupa,
    'answer': final_answer_send,
    'share_link': Link_to_this_colab
}

if Link_to_this_colab == colab_base or Link_to_this_colab == "":
    print("Podaj link do swojego Colab")
    display(Javascript('alert("Podaj link do swojego Colab");'))
else:
    if final_answer == "":
        print("Podaj rozwiązanie")
        display(Javascript('alert("Podaj rozwiązanie");'))
    else:
        response = requests.post(url, json=data)
        if response.status_code == 200:
            response_data = response.json()
            print("Wysłano!")

            points = response_data.get('points', 0)
            if points > 0:
                print(f"Punkty za zadanie: {points}")
            else:
                print("Zadanie wymaga ręcznego zatwierdzenia lub wynik jest niepoprawny")
        else:
            print("błąd wysyłki:", response.status_code)
            print("Response Text:", response.text)


0.432,0.337,0.231,0.319,0.362,0.319,0.243,0.243,0.514
53 znaków
Wysłano!
Punkty za zadanie: 1


#Task C03

## 🧩 Sinusoidalne wektory pozycyjne (positional encoding)

W tym zadaniu zaimplementujesz **sinusoidalne kodowanie pozycji** znane z pracy *Attention is All You Need*.

---

## 🎯 Cel

- zrozumieć formułę sinusoidalnych positional encodings,
- wygenerować macierz kodowań dla kilku pozycji i wymiarów.

---

## 📘 Kontekst

Dla pozycji \(pos\) i wymiaru \(i\) (od 0) oraz rozmiaru modelu \(d_{model}\):

$$
\begin{aligned}
PE(pos, 2i) &= \sin\left( \frac{pos}{10000^{2i/d_{model}}} \right), \\
PE(pos, 2i+1) &= \cos\left( \frac{pos}{10000^{2i/d_{model}}} \right).
\end{aligned}
$$

W praktyce implementujemy to wektorowo dla wszystkich pozycji i wszystkich wymiarów.

---

## ✅ Twoje zadanie

1. W komórce `dane` masz:
   - `max_len = 4`,
   - `d_model = 8`.
2. Zaimplementuj funkcję:

   ```python
   def positional_encoding(max_len, d_model):
       ...
   ```

   która zwraca macierz `PE` o kształcie `(max_len, d_model)` zgodnie z powyższą definicją.
3. W `Answer`:
   - policz `PE = positional_encoding(max_len, d_model)`,
   - wybierz wiersz dla pozycji `pos = 2`,
   - zaokrąglij 8 liczb do 3 miejsc po przecinku,
   - zapisz w `final_answer` jako string:

   ```text
   x0,x1,x2,x3,x4,x5,x6,x7
   ```

bez spacji.


### 🔗 Przydatne funkcje NumPy:
- `np.sin` — https://numpy.org/doc/stable/reference/generated/numpy.sin.html
- `np.cos` — https://numpy.org/doc/stable/reference/generated/numpy.cos.html
- Operacje wektorowe i broadcasting — https://numpy.org/doc/stable/user/basics.broadcasting.html

In [14]:
#@title dane

import numpy as np

max_len = 4
d_model = 8


In [15]:
#@title code

import numpy as np

def positional_encoding(max_len, d_model):
    """Zwraca macierz PE o kształcie (max_len, d_model)."""
    # TODO: zaimplementuj sinusoidalne kodowanie pozycji wg wzoru z Attention is All You Need.
    # Wskazówka:
    # - stwórz wektor positions o kształcie (max_len, 1)
    # - stwórz wektor dims o kształcie (1, d_model//2) z kolejnymi i
    # - policz odpowiednie "div_term" = 10000 ** (2*i/d_model)
    # - wypełnij pary (2i) sinusami, (2i+1) cosinusami
    PE = np.zeros((max_len, d_model))
    positions = np.arange(max_len).reshape(-1, 1)
    dims = np.arange(d_model // 2).reshape(1, -1)
    div_term = 10000 ** (2 * dims / d_model)
    PE[:, 0::2] = np.sin(positions / div_term)
    PE[:, 1::2] = np.cos(positions / div_term)
    return PE


In [20]:
# @title Answer

PE = positional_encoding(max_len, d_model)
row = PE[2]  # pozycja 2
rounded = [f"{x:.3f}" for x in row]
final_answer = ",".join(rounded)
print(final_answer)


0.909,-0.416,0.199,0.980,0.020,1.000,0.002,1.000


In [21]:
#@title Submit answer (final_answer)
import requests
taskID = "TaskC03"
url = 'https://www.duszekjk.com/ugpt/api/submit_answer/'
final_answer_size = len(final_answer)

final_answer_send = final_answer[:300] + "\n" + str(final_answer_size) + " znaków"
print(final_answer_send)
data = {
    'student_id': Student_ID,
    'student_mail': Mail,
    'task': taskID,
    'grupa': Grupa,
    'answer': final_answer_send,
    'share_link': Link_to_this_colab
}

if Link_to_this_colab == colab_base or Link_to_this_colab == "":
    print("Podaj link do swojego Colab")
    display(Javascript('alert("Podaj link do swojego Colab");'))
else:
    if final_answer == "":
        print("Podaj rozwiązanie")
        display(Javascript('alert("Podaj rozwiązanie");'))
    else:
        response = requests.post(url, json=data)
        if response.status_code == 200:
            response_data = response.json()
            print("Wysłano!")

            points = response_data.get('points', 0)
            if points > 0:
                print(f"Punkty za zadanie: {points}")
            else:
                print("Zadanie wymaga ręcznego zatwierdzenia lub wynik jest niepoprawny")
        else:
            print("błąd wysyłki:", response.status_code)
            print("Response Text:", response.text)


0.909,-0.416,0.199,0.980,0.020,1.000,0.002,1.000
48 znaków
Wysłano!
Punkty za zadanie: 1


#Task C04

## 🧩 Residual + LayerNorm — normalizacja wyjścia warstwy

W tym zadaniu zaimplementujesz **Layer Normalization** na wyjściu z residual connection.

---

## 🎯 Cel

- przećwiczyć schemat `x + sublayer(x)` i normalizację po wymiarze cech,
- zobaczyć, jak zmieniają się wartości po LayerNorm.

---

## 📘 Kontekst

W Transformerze typowy blok wygląda tak:

$$
\text{LayerNorm}(x + \text{sublayer}(x)).
$$

Dla pojedynczego tokenu o wektorze \(h\) (np. o rozmiarze \(d\)):

$$
\mu = \frac{1}{d} \sum_j h_j, \quad
\sigma^2 = \frac{1}{d} \sum_j (h_j - \mu)^2, \quad
\hat{h}_j = \frac{h_j - \mu}{\sqrt{\sigma^2 + \epsilon}},
$$

$$
\text{LayerNorm}(h)_j = \gamma_j \hat{h}_j + \beta_j.
$$

---

## ✅ Twoje zadanie

1. W `dane` masz:
   - `x` — wektor wejściowy,
   - `sublayer_out` — wyjście z podwarstwy,
   - `gamma`, `beta` — parametry skalowania i przesunięcia,
   - `eps` — \(\epsilon\) do stabilizacji.
2. Zaimplementuj funkcję:

   ```python
   def layernorm_residual(x, sublayer_out, gamma, beta, eps=1e-5):
       ...
   ```

   która:
   - liczy `h = x + sublayer_out`,
   - liczy LayerNorm po wymiarze cech (1D),
   - zwraca wektor o tym samym kształcie.
3. W `Answer`:
   - policz `y = layernorm_residual(...)`,
   - zaokrąglij wszystkie wartości do 3 miejsc po przecinku,
   - zapisz w `final_answer` jako string:

   ```text
   y0,y1,y2,y3
   ```

bez spacji.


### 🔗 Przydatne funkcje NumPy:
- `np.mean` — https://numpy.org/doc/stable/reference/generated/numpy.mean.html
- `np.var` — https://numpy.org/doc/stable/reference/generated/numpy.var.html
- `np.sqrt` — https://numpy.org/doc/stable/reference/generated/numpy.sqrt.html

In [22]:
#@title dane

import numpy as np

x = np.array([0.5, -0.5, 1.0, 0.0], dtype=float)
sublayer_out = np.array([0.1, 0.2, -0.3, 0.0], dtype=float)

gamma = np.array([1.0, 1.0, 1.0, 1.0], dtype=float)
beta  = np.array([0.0, 0.0, 0.0, 0.0], dtype=float)

eps = 1e-5


In [23]:
#@title code

import numpy as np

def layernorm_residual(x, sublayer_out, gamma, beta, eps=1e-5):
    """Zwraca LayerNorm(x + sublayer_out) dla jednego wektora."""
    h = x + sublayer_out
    mean = np.mean(h)
    var = np.var(h)
    h_norm = (h - mean) / np.sqrt(var + eps)
    output = gamma * h_norm + beta
    return output


In [24]:
# @title Answer

y = layernorm_residual(x, sublayer_out, gamma, beta, eps=eps)
rounded = [f"{v:.3f}" for v in y]
final_answer = ",".join(rounded)
print(final_answer)


0.843,-1.324,1.083,-0.602


In [25]:
#@title Submit answer (final_answer)
import requests
taskID = "TaskC04"
url = 'https://www.duszekjk.com/ugpt/api/submit_answer/'
final_answer_size = len(final_answer)

final_answer_send = final_answer[:300] + "\n" + str(final_answer_size) + " znaków"
print(final_answer_send)
data = {
    'student_id': Student_ID,
    'student_mail': Mail,
    'task': taskID,
    'grupa': Grupa,
    'answer': final_answer_send,
    'share_link': Link_to_this_colab
}

if Link_to_this_colab == colab_base or Link_to_this_colab == "":
    print("Podaj link do swojego Colab")
    display(Javascript('alert("Podaj link do swojego Colab");'))
else:
    if final_answer == "":
        print("Podaj rozwiązanie")
        display(Javascript('alert("Podaj rozwiązanie");'))
    else:
        response = requests.post(url, json=data)
        if response.status_code == 200:
            response_data = response.json()
            print("Wysłano!")

            points = response_data.get('points', 0)
            if points > 0:
                print(f"Punkty za zadanie: {points}")
            else:
                print("Zadanie wymaga ręcznego zatwierdzenia lub wynik jest niepoprawny")
        else:
            print("błąd wysyłki:", response.status_code)
            print("Response Text:", response.text)


0.843,-1.324,1.083,-0.602
25 znaków
Wysłano!
Punkty za zadanie: 1


#Task C05

## 🧩 Złożoność O(n²) vs okno — liczenie liczby operacji

W tym zadaniu policzysz **przybliżoną liczbę operacji mnożenia** potrzebnych do obliczenia attention
dla pełnej uwagi oraz dla uwagi okienkowej.

---

## 🎯 Cel

- zrozumieć, jak rośnie koszt obliczeniowy attention z długością sekwencji,
- policzyć przybliżoną liczbę mnożeń dla dwóch scenariuszy.

---

## 📘 Kontekst

Dla uproszczonego modelu:

- pełna uwaga: liczba mnożeń \(\approx n^2 \cdot d_k\),
- uwaga okienkowa o szerokości `w`: \(\approx n \cdot w \cdot d_k\),

gdzie:
- `n` — długość sekwencji (liczba tokenów),
- `d_k` — wymiar klucza/zapytania,
- `w` — szerokość okna (ile sąsiednich tokenów bierzemy pod uwagę).

---

## ✅ Twoje zadanie

1. W `dane` masz:
   - konkretne wartości `n`, `d_k`, `w`.
2. Zaimplementuj funkcje:

   ```python
   def flops_full_attention(n, d_k):
       ...
   def flops_window_attention(n, d_k, w):
       ...
   ```

   które zwracają **liczbę mnożeń** jako liczbę całkowitą.
3. W `Answer`:
   - policz `full = flops_full_attention(n, d_k)`,
   - policz `window = flops_window_attention(n, d_k, w)`,
   - zapisz w `final_answer` string:

   ```text
   full=LICZBA1;window=LICZBA2
   ```

bez spacji.


### 🔗 Przydatne funkcje NumPy:
- Operacje arytmetyczne w NumPy — https://numpy.org/doc/stable/user/quickstart.html#simple-arithmetic
- Potęgowanie `**` — https://numpy.org/doc/stable/reference/generated/numpy.power.html

In [26]:
#@title dane

# Uproszczone parametry
n = 512    # długość sekwencji
d_k = 64   # wymiar klucza/zapytania
w = 32     # szerokość okna w uwadze okienkowej


In [27]:
#@title code

def flops_full_attention(n, d_k):
    """Przybliżona liczba mnożeń dla pełnej uwagi: n^2 * d_k."""
    return n * n * d_k

def flops_window_attention(n, d_k, w):
    """Przybliżona liczba mnożeń dla uwagi okienkowej: n * w * d_k."""
    return n * w * d_k

In [28]:
# @title Answer

full = flops_full_attention(n, d_k)
window = flops_window_attention(n, d_k, w)

final_answer = f"full={full};window={window}"
print(final_answer)

full=16777216;window=1048576


In [29]:
#@title Submit answer (final_answer)
import requests
taskID = "TaskC05"
url = 'https://www.duszekjk.com/ugpt/api/submit_answer/'
final_answer_size = len(final_answer)

final_answer_send = final_answer[:300] + "\n" + str(final_answer_size) + " znaków"
print(final_answer_send)
data = {
    'student_id': Student_ID,
    'student_mail': Mail,
    'task': taskID,
    'grupa': Grupa,
    'answer': final_answer_send,
    'share_link': Link_to_this_colab
}

if Link_to_this_colab == colab_base or Link_to_this_colab == "":
    print("Podaj link do swojego Colab")
    display(Javascript('alert("Podaj link do swojego Colab");'))
else:
    if final_answer == "":
        print("Podaj rozwiązanie")
        display(Javascript('alert("Podaj rozwiązanie");'))
    else:
        response = requests.post(url, json=data)
        if response.status_code == 200:
            response_data = response.json()
            print("Wysłano!")

            points = response_data.get('points', 0)
            if points > 0:
                print(f"Punkty za zadanie: {points}")
            else:
                print("Zadanie wymaga ręcznego zatwierdzenia lub wynik jest niepoprawny")
        else:
            print("błąd wysyłki:", response.status_code)
            print("Response Text:", response.text)


full=16777216;window=1048576
28 znaków
Wysłano!
Punkty za zadanie: 1


#Task C06

## 🧩 Dropout na wagach attention

W tym zadaniu zaimplementujesz **dropout na wektorze wag attention**, a następnie ponownie
znormalizujesz wagi tak, aby sumowały się do 1.

---

## 🎯 Cel

- zrozumieć, jak działa dropout w attention,
- poćwiczyć losowanie maski i renormalizację.

---

## 📘 Kontekst

Typowy dropout na attention działa tak:

1. Mamy wektor wag \(a\) (np. długości 4), taki że \(\sum_j a_j = 1\).
2. Losujemy maskę Bernoulliego (np. z prawdopodobieństwem zachowania \(p_{keep}\)).
3. Wagi, dla których maska = 0, zerujemy.
4. Jeśli cokolwiek zostało, renormalizujemy tak, aby suma znów była równa 1.

---

## ✅ Twoje zadanie

1. W `dane` masz:
   - `attn = np.array([...])` — wektor wag,
   - `p_keep` — prawdopodobieństwo zachowania wagi,
   - `seed` — ziarno losowości.
2. Zaimplementuj funkcję:

   ```python
   def dropout_and_renorm(attn, p_keep, seed):
       ...
   ```

   która:
   - ustawia `np.random.seed(seed)`,
   - losuje maskę Bernoulliego o tym samym kształcie co `attn`,
   - ustawia wyzerowane wagi tam, gdzie maska = 0,
   - jeśli suma pozostałych wag > 0:
     - dzieli przez sumę, żeby znów sumowały się do 1,
   - jeśli wszystkie wagi zostały wyzerowane:
     - zwraca wektor równomierny (wszystkie elementy = 1/len(attn)).
3. W `Answer`:
   - policz `attn_out = dropout_and_renorm(attn, p_keep, seed)`,
   - zaokrąglij do 3 miejsc po przecinku,
   - zapisz w `final_answer` jako string:

   ```text
   a0,a1,a2,a3
   ```

bez spacji.


### 🔗 Przydatne funkcje NumPy:
- `np.random.seed` — https://numpy.org/doc/stable/reference/random/generated/numpy.random.seed.html
- `np.random.rand` — https://numpy.org/doc/stable/reference/random/generated/numpy.random.rand.html
- Maskowanie tablic — https://numpy.org/doc/stable/user/basics.indexing.html

In [30]:
#@title dane

import numpy as np

attn = np.array([0.1, 0.2, 0.3, 0.4], dtype=float)
p_keep = 0.5
seed = 123  # ziarno dla powtarzalności


In [ ]:
#@title code

import numpy as np

def dropout_and_renorm(attn, p_keep, seed):
    """Dropout + renormalizacja wektora wag attention."""
    if seed is not None:
        np.random.seed(seed)
    
    mask = np.random.rand(*attn.shape) < p_keep
    
    dropped = attn * mask
    
    total = dropped.sum()
    if total > 0:
        return dropped / total
    else:
        return np.ones_like(attn) / len(attn)


In [35]:
# @title Answer

attn_out = dropout_and_renorm(attn, p_keep, seed)
rounded = [f"{v:.3f}" for v in attn_out]
final_answer = ",".join(rounded)
print(final_answer)

0.000,0.400,0.600,0.000


In [36]:
#@title Submit answer (final_answer)
import requests
taskID = "TaskC06"
url = 'https://www.duszekjk.com/ugpt/api/submit_answer/'
final_answer_size = len(final_answer)

final_answer_send = final_answer[:300] + "\n" + str(final_answer_size) + " znaków"
print(final_answer_send)
data = {
    'student_id': Student_ID,
    'student_mail': Mail,
    'task': taskID,
    'grupa': Grupa,
    'answer': final_answer_send,
    'share_link': Link_to_this_colab
}

if Link_to_this_colab == colab_base or Link_to_this_colab == "":
    print("Podaj link do swojego Colab")
    display(Javascript('alert("Podaj link do swojego Colab");'))
else:
    if final_answer == "":
        print("Podaj rozwiązanie")
        display(Javascript('alert("Podaj rozwiązanie");'))
    else:
        response = requests.post(url, json=data)
        if response.status_code == 200:
            response_data = response.json()
            print("Wysłano!")

            points = response_data.get('points', 0)
            if points > 0:
                print(f"Punkty za zadanie: {points}")
            else:
                print("Zadanie wymaga ręcznego zatwierdzenia lub wynik jest niepoprawny")
        else:
            print("błąd wysyłki:", response.status_code)
            print("Response Text:", response.text)


0.000,0.400,0.600,0.000
23 znaków
Wysłano!
Punkty za zadanie: 1
